# 01 — Data Exploration: OSCD Satellite Change Detection

This notebook explores the OSCD (Onera Satellite Change Detection) dataset:
- Load samples with RGB + NIR bands
- Compute NDVI for before/after images
- Visualize change masks and class imbalance
- Verify tensor shapes and value ranges

In [ ]:
!pip install torchgeo matplotlib numpy tqdm -q

In [ ]:
import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt
from torchgeo.datasets import OSCD100

print("OSCD100 dataset loaded successfully")

## 1. Download and Load OSCD100

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'oscd100')

dataset = OSCD100(
    root=root,
    split='train',
    bands=OSCD100.rgb_bands,
    download=True,
)

print(f"Training samples: {len(dataset)}")
print(f"Bands: {OSCD100.rgb_bands}")

## 2. Inspect Sample Structure

In [ ]:
sample = dataset[0]

print("Sample keys:", list(sample.keys()))
print(f"Image shape: {sample['image'].shape}")
print(f"Mask shape: {sample['mask'].shape}")
print(f"Image dtype: {sample['image'].dtype}")
print(f"Mask dtype: {sample['mask'].dtype}")
print(f"Image range: [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")
print(f"Mask unique values: {sample['mask'].unique().tolist()}")

## 3. Visualize Before/After and Change Mask

In [ ]:
fig = dataset.plot(sample, suptitle="OSCD100 Sample")
plt.show()

In [ ]:
def visualize_sample(dataset, idx=0):
    sample = dataset[idx]
    image = sample['image'].numpy()  # [2, C, H, W] or [C, H, W]
    mask = sample['mask'].numpy()

    if image.ndim == 4:
        t1 = image[0]  # [C, H, W]
        t2 = image[1]
    else:
        t1 = image
        t2 = image

    t1_rgb = t1[:3].transpose(1, 2, 0)
    t2_rgb = t2[:3].transpose(1, 2, 0)

    t1_rgb = np.clip(t1_rgb / t1_rgb.max(), 0, 1) if t1_rgb.max() > 0 else t1_rgb
    t2_rgb = np.clip(t2_rgb / t2_rgb.max(), 0, 1) if t2_rgb.max() > 0 else t2_rgb

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(t1_rgb)
    axes[0].set_title('Before (T1)')
    axes[0].axis('off')

    axes[1].imshow(t2_rgb)
    axes[1].set_title('After (T2)')
    axes[1].axis('off')

    mask_2d = mask.squeeze()
    axes[2].imshow(t2_rgb, alpha=0.6)
    change_overlay = np.zeros((*mask_2d.shape, 3))
    change_overlay[mask_2d > 0] = [1, 0, 0]
    axes[2].imshow(change_overlay, alpha=0.5)
    axes[2].set_title('Change Mask')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

visualize_sample(dataset, idx=0)

## 4. Compute NDVI for Before/After

In [ ]:
def compute_ndvi(nir, red, eps=1e-10):
    nir = nir.astype(np.float32)
    red = red.astype(np.float32)
    return np.clip((nir - red) / (nir + red + eps), -1, 1)

sample = dataset[0]
image = sample['image'].numpy()

if image.ndim == 4:
    t1 = image[0]
    t2 = image[1]
else:
    t1 = image
    t2 = image

print(f"Bands available: {t1.shape[0]}")
if t1.shape[0] >= 4:
    ndvi_t1 = compute_ndvi(t1[3], t1[0])  # NIR=B08(idx3), Red=B04(idx0)
    ndvi_t2 = compute_ndvi(t2[3], t2[0])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    im1 = axes[0].imshow(ndvi_t1, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[0].set_title('NDVI Before')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046)

    im2 = axes[1].imshow(ndvi_t2, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1].set_title('NDVI After')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046)

    ndvi_diff = ndvi_t2 - ndvi_t1
    im3 = axes[2].imshow(ndvi_diff, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    axes[2].set_title('NDVI Change')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046)

    plt.suptitle('NDVI Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {t1.shape[0]} bands available, need at least 4 for NDVI")

## 5. Class Imbalance Analysis

In [ ]:
total_pixels = 0
changed_pixels = 0

for i in range(len(dataset)):
    mask = dataset[i]['mask'].numpy().squeeze()
    total_pixels += mask.size
    changed_pixels += (mask > 0).sum()

ratio = changed_pixels / total_pixels
print(f"Total pixels: {total_pixels:,}")
print(f"Changed pixels: {changed_pixels:,}")
print(f"Change ratio: {ratio:.4f} ({ratio*100:.2f}%)")
print(f"Class imbalance ratio (no-change:change): {(1-ratio)/ratio:.1f}:1")

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.bar(['No Change', 'Changed'], [1-ratio, ratio], color=['#2ecc71', '#e74c3c'])
ax.set_ylabel('Fraction of Pixels')
ax.set_title(f'Class Imbalance: {ratio*100:.1f}% changed')
plt.tight_layout()
plt.show()

## 6. Multiple Samples Overview

In [ ]:
n_show = min(5, len(dataset))
fig, axes = plt.subplots(n_show, 3, figsize=(15, 4*n_show))

for i in range(n_show):
    sample = dataset[i]
    image = sample['image'].numpy()
    mask = sample['mask'].numpy().squeeze()

    if image.ndim == 4:
        t1_rgb = image[0, :3].transpose(1, 2, 0)
        t2_rgb = image[1, :3].transpose(1, 2, 0)
    else:
        t1_rgb = image[:3].transpose(1, 2, 0)
        t2_rgb = t1_rgb

    t1_rgb = np.clip(t1_rgb / (t1_rgb.max() + 1e-10), 0, 1)
    t2_rgb = np.clip(t2_rgb / (t2_rgb.max() + 1e-10), 0, 1)

    axes[i, 0].imshow(t1_rgb)
    axes[i, 0].set_title(f'Sample {i}: Before')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(t2_rgb)
    axes[i, 1].set_title(f'Sample {i}: After')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(mask, cmap='gray')
    axes[i, 2].set_title(f'Sample {i}: Change ({mask.sum():.0f} px)')
    axes[i, 2].axis('off')

plt.suptitle('OSCD Dataset Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Band Statistics

In [ ]:
band_names = ['Red', 'Green', 'Blue']
if len(OSCD100.rgb_bands) > 3:
    band_names.extend([OSCD100.rgb_bands[i] for i in range(3, len(OSCD100.rgb_bands))])

all_means = []
all_stds = []

for i in range(min(20, len(dataset))):
    img = dataset[i]['image'].numpy()
    if img.ndim == 4:
        img = img.reshape(img.shape[0]*img.shape[1], img.shape[2], img.shape[3])
    for b in range(min(img.shape[0], len(band_names))):
        all_means.append(img[b].mean())
        all_stds.append(img[b].std())

print("Per-band statistics (across first 20 samples):")
for b in range(min(3, len(band_names))):
    band_means = [all_means[i] for i in range(b, len(all_means), len(band_names))]
    band_stds = [all_stds[i] for i in range(b, len(all_stds), len(band_names))]
    print(f"  {band_names[b]}: mean={np.mean(band_means):.4f}, std={np.mean(band_stds):.4f}")

## Summary

- **OSCD100** contains 100 pre-cropped 256x256 Sentinel-2 image pairs
- Each image has 3 RGB bands (B04, B03, B02) at 10m resolution
- Change masks are binary: 0=no change, 255=change
- **Class imbalance**: Changed pixels are typically ~3-8% of total (need BCE+Dice loss)
- NDVI adds domain knowledge: `(NIR - Red) / (NIR + Red)` highlights vegetation change
- Next: `02_train_supervised.ipynb` to train the Siamese U-Net